In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

pio.renderers.default = 'vscode'

In [3]:
# import preprocessed file

df = pd.DataFrame(pd.read_csv('zomato_preprocessed_dataset.csv'))
df.head()

,name,online_order,book_table,rate,votes,location,rest_type,cuisines,price
0,Jalsa,1,1,4.1,775,Banashankari,Casual Dining,North Indian,800
1,Spice Elephant,1,0,4.1,787,Banashankari,Casual Dining,Asian,800
2,San Churro Cafe,1,0,3.8,918,Banashankari,Cafe,Cafe,800
3,Addhuri Udupi Bhojana,0,0,3.7,88,Banashankari,Quick Bites,South Indian,300
4,Grand Village,0,0,3.8,166,Basavanagudi,Casual Dining,North Indian,600


In [6]:
# 1. How many restaurants are present for each cuisine

fig = px.histogram(
    df, x="cuisines", color="cuisines", title="Restaurants for each cuisine"
)
fig.show()

In [9]:
# 2. What is the distribution of restaurant ratings?

fig = px.histogram(df, x='rate', title='Distribution of restaurant', color = 'rate')
fig.show()

In [26]:
# 2. Which locations have the highest number of restaurants?

location_percent = df["location"].value_counts().reset_index()

fig = px.bar(
    location_percent,
    x="location",
    y="count",
    color="location",
    title="Highest no.of.restaurants in location",
    width=1200,
    height=600,
)

fig.show()

In [35]:
# 4. Which cuisines are most popular?

cuisine_df = df["cuisines"].value_counts().reset_index()
cuisine_df

px.pie(
    cuisine_df, values="count", color="cuisines", title="Most popular cuisines", names='cuisines'
).update_traces(textposition="inside").show()

In [40]:
# 5. Do restaurants with online ordering have higher ratings?

avg_rating = df.groupby(["online_order"])["rate"].mean().reset_index()

px.bar(
    avg_rating,
    x="online_order",
    y="rate",
    color="online_order",
    title="Average Rating vs Online Ordering",
).show()

In [56]:
# Q6) What is the Avg Price Distibution of highest rated restaurant for each Cuisine Type?

avg_price = (
    df.sort_values(by="rate", ascending=False)
    .groupby(by=["location", "cuisines"])["price"]
    .mean()
    .reset_index()
)

print(avg_price)

px.scatter(
    avg_price,
    x="location",
    y="price",
    color="cuisines",
    title="Avg Price Distibution of highest rated restaurant",
    symbol="cuisines",
    height=800,
    width=1200,
).show()

        location      cuisines        price
0            BTM         Asian   438.888889
1            BTM     Beverages   234.756098
2            BTM          Cafe   376.428571
3            BTM   Continental   639.908257
4            BTM   East Indian   298.684211
..           ...           ...          ...
676  Yeshwantpur   East Indian   150.000000
677  Yeshwantpur     Fast Food   311.111111
678  Yeshwantpur  North Indian   445.000000
679  Yeshwantpur       Seafood  1200.000000
680  Yeshwantpur  South Indian   376.000000

[681 rows x 3 columns]


In [59]:
# Q7) Which areas have a large number of continental Restaurants?

continental_df = df[df['cuisines'] == 'Continental']
continental_df.head(5)

,name,online_order,book_table,rate,votes,location,rest_type,cuisines,price
22,My Tea House,1,0,3.6,62,Banashankari,Quick Bites,Continental,600
52,Roving Feast,0,0,4.0,1047,Banashankari,Quick Bites,Continental,450
58,Peppy Peppers,0,0,4.2,244,Banashankari,Casual Dining,Continental,800
65,Gustoes Beer House,0,0,4.1,868,Banashankari,Bar,Continental,1200
68,The Good Bowl,1,0,3.6,6,Banashankari,Delivery/Takeaway,Continental,500


In [65]:
px.histogram(
    continental_df,
    x="location",
    color="location",
    title="Location with large number of continental Restaurants",
    height = 600
).show()

In [66]:
# Q8) Is there a relation between Price and Rating by each Cuisine Type?

relation = df.groupby(['cuisines', 'rate'])['price'].mean().reset_index()
relation

,cuisines,rate,price
0,Asian,0.0,380.257511
1,Asian,2.3,650.000000
2,Asian,2.4,1000.000000
3,Asian,2.5,810.000000
4,Asian,2.6,500.000000
...,...,...,...
244,West Indian,4.2,1133.333333
245,West Indian,4.4,1400.000000
246,West Indian,4.5,1800.000000
247,West Indian,4.6,1500.000000


In [70]:
px.line(relation, y="price", x="rate",color='cuisines').show()

In [71]:
# Is there a relation between Region and Price?


fig = px.scatter(
    df.groupby(["location"])["price"].mean().reset_index(),
    x="location",
    y="price",
    height=700
).update_traces(marker_size=8)
fig.update_yaxes(range=[0, 2000])
fig.show()

In [75]:
# Q10) Find the list of Affordable Restaurants?

# criteria => low price and high rated restaurant
# we'll calculate the economical price (i.e.. less than1/4th of expensive price).

high_rated = df.sort_values(by='rate', ascending=False)

aff_price = df["price"].max() / 4

aff_res = high_rated[high_rated["price"] <= aff_price]
print("total affordable restaurants: ", aff_res["name"].count())

aff_res.sort_values(by=["rate", "price"], ascending=[False, True]).head(5)

# Normalization
aff_res["rate_norm"] = aff_res["rate"] / aff_res["rate"].max()
aff_res["price_norm"] = aff_res["price"] / aff_res["price"].max()

# adding weights (70% to rating and 30% to price)
# dividing by 1000 because rating scale is 0-5 but price is 0-5000.
aff_res["score"] = (0.7 * aff_res["rate_norm"]) + (0.3 * (1- aff_res["price_norm"]))

aff_res.sort_values(by="score", ascending=False)



total affordable restaurants:  22083


,name,online_order,book_table,rate,votes,location,rest_type,cuisines,price,rate_norm,price_norm,score
2045,Brahmin's Coffee Bar,0,0,4.8,2679,Basavanagudi,Quick Bites,South Indian,100,0.979592,0.066667,0.965714
20721,CTR,1,0,4.8,4421,Malleshwaram,Quick Bites,South Indian,150,0.979592,0.100000,0.955714
512,Taaza Thindi,0,0,4.7,651,Banashankari,Quick Bites,South Indian,100,0.959184,0.066667,0.951429
20821,O.G. Variar & Sons,0,0,4.8,1161,Rajajinagar,Quick Bites,Fast Food,200,0.979592,0.133333,0.945714
17729,O.G. Variar & Sons,0,0,4.8,1156,Rajajinagar,Quick Bites,Fast Food,200,0.979592,0.133333,0.945714
...,...,...,...,...,...,...,...,...,...,...,...,...
1658,Levitate Brewery and Kitchen,0,1,0.0,0,JP Nagar,Bar,Fast Food,1500,0.000000,1.000000,0.000000
9994,Blu Oyster,0,1,0.0,0,Indiranagar,Casual Dining,Seafood,1500,0.000000,1.000000,0.000000
19174,Copacabana Pub,1,0,0.0,0,Ulsoor,Bar,Continental,1500,0.000000,1.000000,0.000000
16101,The Flying Trapeze,0,1,0.0,0,Koramangala 5th Block,Casual Dining,Continental,1500,0.000000,1.000000,0.000000


In [77]:
# Q10) Find the list of most Reliable Restaurants?
# The criteria for most Reliable Restaurants would be:-
# 1) Low Price 2) High Rated 3) Large No. of Votes

mean_votes = df['votes'].mean()

# Finding list of restaurants that have Votes greater than and equal to Mean of Vote
mean_rest_df = df[['name', 'price', 'cuisines', 'location', 'rest_type', 'votes']]
mean_rest_df = mean_rest_df[mean_rest_df['votes'] > int(mean_votes)]
mean_rest_df.sort_values(by='votes', inplace=True)
mean_rest_df

,name,price,cuisines,location,rest_type,votes
10524,Prems Graama Bhojanam,400,South Indian,Jayanagar,Casual Dining,326
19920,Dindigul Thalappakatti,800,South Indian,Indiranagar,Casual Dining,326
20472,The Paramount Hotel,1000,North Indian,Domlur,Casual Dining,326
20462,Vembanad - The Paul,1800,Seafood,Domlur,Casual Dining,326
14805,The Himalayan,1200,Asian,Koramangala 5th Block,Casual Dining,326
...,...,...,...,...,...,...
15583,Truffles,900,Cafe,Koramangala 5th Block,Cafe,14723
16039,Truffles,900,Cafe,Koramangala 5th Block,Cafe,14726
9960,Toit,1500,Continental,Indiranagar,Bar,14956
2375,Byg Brewski Brewing Company,1600,Continental,Sarjapur Road,Bar,16345


In [79]:
reliable_rest_df = pd.merge(
    mean_rest_df, high_rated, how="inner", on=["name", "location"]
)
reliable_rest_df = reliable_rest_df[
    ["name", "price_x", "cuisines_x", "location", "rest_type_x"]
]
reliable_rest_df.rename(
    columns={
        "NAME": "name",
        "PRICE_x": "price",
        "CUSINE_CATEGORY_x": "cuisines",
        "location": "location",
        "rest_type_x": "rest_type",
    },
    inplace=True,
)
reliable_rest_df

,name,price_x,cuisines_x,location,rest_type
0,Prems Graama Bhojanam,400,South Indian,Jayanagar,Casual Dining
1,Prems Graama Bhojanam,400,South Indian,Jayanagar,Casual Dining
2,Prems Graama Bhojanam,400,South Indian,Jayanagar,Casual Dining
3,Prems Graama Bhojanam,400,South Indian,Jayanagar,Casual Dining
4,Dindigul Thalappakatti,800,South Indian,Indiranagar,Casual Dining
...,...,...,...,...,...
20588,Toit,1500,Continental,Indiranagar,Bar
20589,Byg Brewski Brewing Company,1600,Continental,Sarjapur Road,Bar
20590,Byg Brewski Brewing Company,1600,Continental,Sarjapur Road,Bar
20591,Byg Brewski Brewing Company,1600,Continental,Sarjapur Road,Bar
